# Kaplan-Meier Estimation: Unemployment Duration

This notebook demonstrates nonparametric survival analysis using the **UnempDur** dataset,
the standard packaged version of data used in:

> Meyer, B.D. (1990). *Unemployment Insurance and Unemployment Spells*. **Econometrica**, 58(4), 757–782.

The key question: do unemployment insurance (UI) recipients stay unemployed longer than non-recipients?
We use Kaplan-Meier survival curves to visualize this, then apply a log-rank test and a Cox model.

## 1. Setup

In [ ]:
# Install lifelines if needed
# !pip install lifelines

import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['figure.dpi'] = 110

## 2. Load the Data

The dataset is hosted on the [Rdatasets mirror](https://vincentarelbundock.github.io/Rdatasets/)
and can be read directly into pandas.

| Variable | Description |
|----------|-------------|
| `spell`  | Unemployment spell duration (weeks) |
| `censor1` | 1 = found a job (event); 0 = censored |
| `ui`     | Received unemployment insurance? (`yes`/`no`) |
| `age`    | Age in years |
| `logwage` | Log of prior wage |
| `tenure` | Tenure at previous job (years) |
| `reprate` | UI replacement rate (benefit / prior wage) |
| `disrate` | UI disregard rate |

In [ ]:
url = "https://vincentarelbundock.github.io/Rdatasets/csv/Ecdat/UnempDur.csv"
unemp_df = pd.read_csv(url, index_col=0)

print(f"Observations: {len(unemp_df)}")
print(f"Events (found job): {unemp_df['censor1'].sum()}  "
      f"({unemp_df['censor1'].mean()*100:.1f}%)")
print(f"UI recipients: {(unemp_df['ui']=='yes').sum()}  "
      f"({(unemp_df['ui']=='yes').mean()*100:.1f}%)")
unemp_df.head()

In [ ]:
cox_features_df = unemp_df.assign(
    ui_binary=(df['ui'] == 'yes').astype(int)
)[['spell', 'censor1', 'ui_binary', 'age', 'logwage', 'tenure', 'reprate']]

cox_summary = cox_features_df.describe().T
cox_summary['missing'] = cox_features_df.isna().sum()
cox_summary['missing_pct'] = (100 * cox_summary['missing'] / len(cox_features_df)).round(2)

cox_summary = cox_summary[
    ['count', 'missing', 'missing_pct', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
].round(3)

cox_summary

## 3. Kaplan-Meier Survival Curves

The **survival function** S(t) estimates the probability of remaining unemployed beyond week *t*.
We stratify by UI receipt to replicate the key comparison in Meyer (1990).

In [ ]:
fig, ax = plt.subplots()

colors = {'yes': '#d62728', 'no': '#1f77b4'}
labels = {'yes': 'UI recipient', 'no': 'No UI'}

kmf = KaplanMeierFitter()
for group in ['yes', 'no']:
    mask = unemp_df['ui'] == group
    kmf.fit(
        unemp_df.loc[mask, 'spell'],
        event_observed=unemp_df.loc[mask, 'censor1'],
        label=labels[group]
    )
    kmf.plot_survival_function(ax=ax, color=colors[group], ci_show=True)

ax.set_xlabel('Weeks unemployed')
ax.set_ylabel('S(t): P(still unemployed)')
ax.set_title('Kaplan-Meier Survival Curves by UI Receipt\n(Meyer 1990 / UnempDur)')
ax.legend()
plt.tight_layout()
plt.show()

- **UI recipients stay unemployed longer:** The red curve sits above the blue at every duration, meaning UI recipients have a higher probability of still being unemployed at any given week.
- This is the central visual finding in Meyer (1990) — UI receipt appears to extend unemployment spells, consistent with a moral hazard effect.
- The shaded bands are 95% confidence intervals; they overlap somewhat at longer durations where sample sizes shrink.

## 4. Log-Rank Test

The log-rank test asks: are the two survival curves statistically different?

In [ ]:
ui_yes = unemp_df[unemp_df['ui'] == 'yes']
ui_no  = unemp_df[unemp_df['ui'] == 'no']

result = logrank_test(
    ui_yes['spell'], ui_no['spell'],
    event_observed_A=ui_yes['censor1'],
    event_observed_B=ui_no['censor1']
)
result.print_summary()

- **Statistically significant difference:** A low p-value (typically < 0.05) confirms the two survival curves are not just visually different — the difference is unlikely due to chance.
- The log-rank test is nonparametric and makes no distributional assumptions; it weights all time points equally when comparing the two groups.

## 5. Median Unemployment Duration

The median spell length (week at which S(t) = 0.5) is a useful summary statistic.

In [ ]:
for group in ['yes', 'no']:
    mask = unemp_df['ui'] == group
    kmf.fit(
        unemp_df.loc[mask, 'spell'],
        event_observed=unemp_df.loc[mask, 'censor1'],
        label=labels[group]
    )
    median = kmf.median_survival_time_
    print(f"{labels[group]:15s}  median spell = {median:.0f} weeks")

- **Median as a concrete summary:** The median spell is the week by which half the group has found a job; it's more robust than the mean for skewed duration data.
- A longer median for UI recipients quantifies the gap suggested by the survival curves — translating the visual difference into a single interpretable number.

## 6. Cox Proportional Hazards Model

The Cox model lets us control for covariates while estimating the hazard ratio for UI receipt.
A hazard ratio < 1 for UI means lower instantaneous exit rate from unemployment (longer spells).

In [ ]:
# Encode UI as binary
df_cox = unemp_df.copy()
df_cox['ui_binary'] = (df_cox['ui'] == 'yes').astype(int)

covariates = ['ui_binary', 'age', 'logwage', 'tenure', 'reprate']
cox_df = df_cox[['spell', 'censor1'] + covariates].dropna()

cph = CoxPHFitter()
cph.fit(cox_df, duration_col='spell', event_col='censor1')
cph.print_summary()

*Comments*

- **Controls for confounders:** Unlike the raw KM comparison, the Cox model adjusts for age, prior wage, job tenure, and the replacement rate simultaneously — isolating the independent effect of UI receipt.
- A hazard ratio < 1 for `ui_binary` means UI recipients exit unemployment more slowly (lower instantaneous job-finding rate), consistent with moral hazard.
- Coefficients for other covariates (e.g., `logwage`, `tenure`) reveal how worker characteristics independently predict job-finding speed.

## 7. Visualize Hazard Ratios

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
cph.plot(ax=ax)
ax.set_title('Cox Model: Hazard Ratios with 95% CI')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

- **Reading the forest plot:** Each dot is a hazard ratio and the horizontal bar is its 95% CI; points to the left of the dashed line (HR < 1) indicate a lower hazard (slower exit from unemployment).
- Variables whose confidence intervals cross zero (on the log scale) are not statistically significant at the 5% level.

## References

- Meyer, B.D. (1990). Unemployment Insurance and Unemployment Spells. *Econometrica*, 58(4), 757–782.
- Kiefer, N.M. (1988). Economic Duration Data and Hazard Functions. *Journal of Economic Literature*, 26(2), 646–679.
- Davidson-Pilon, C. (2019). *lifelines*: Survival analysis in Python. JOSS. https://lifelines.readthedocs.io
- Data via Rdatasets: https://vincentarelbundock.github.io/Rdatasets/csv/Ecdat/UnempDur.csv